# PCA — Country Data (Help International)
**Lead University · Minería de Datos · Tarea 3**

Análisis de Componentes Principales sobre indicadores de desarrollo.
Objetivo: segmentar países para priorización de ayuda humanitaria e identificar contrastes socioeconómicos.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scripts import PCAAnalysis

sns.set(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
print('Librerías cargadas.')

In [ ]:
df = pd.read_csv('datos/Country-data.csv')
print(f'Registros: {df.shape[0]} | Variables: {df.shape[1]}')
print(f'Columnas: {df.columns.tolist()}')
df.head()

---
## Limpieza y selección de variables

Se usan todas las variables numéricas de indicadores de desarrollo. La columna `country` se conserva como índice para etiquetar países en los gráficos.

In [ ]:
num_cols = ['child_mort', 'exports', 'health', 'imports', 'income',
            'inflation', 'life_expec', 'total_fer', 'gdpp']

df_clean = df.dropna(subset=num_cols).copy()
df_clean = df_clean.set_index('country')
print(f'Registros tras limpieza: {len(df_clean)}')
print()
print(df_clean[num_cols].describe().round(2))

---
## Ajuste del PCA

In [ ]:
pca = PCAAnalysis(n_components=5)
pca.ajustar(df_clean, columnas_pca=num_cols)

print('Varianza explicada por componente:')
for i, v in enumerate(pca.varianza_explicada):
    acum = pca.varianza_acumulada[i]
    print(f'  PC{i+1}: {v:.2f} % (acumulada: {acum:.2f} %)')
print()
print(f'Componentes necesarios para >= 80%: {pca.n_componentes_80()}')

---
## Visualizaciones obligatorias

### Scree Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pca.scree_plot(ax=ax)
plt.tight_layout()
plt.show()

**Interpretación:**

- El primer componente captura casi la mitad de la varianza, representando el eje de desarrollo (riqueza vs pobreza).
- Se necesitan 4 componentes para alcanzar el 80 %, reflejando que los indicadores capturan múltiples dimensiones del desarrollo (económica, demográfica, comercial).

### Círculo de correlación

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
pca.circulo_correlacion(ax=ax)
plt.tight_layout()
plt.show()

**Interpretación:**

- `child_mort` y `total_fer` (mortalidad infantil y fertilidad) se oponen diametralmente a `gdpp`, `income` y `life_expec`, formando el eje de desarrollo humano en PC1.
- `exports` e `imports` definen el PC2 (apertura comercial), ortogonal al desarrollo.
- `inflation` y `health` tienen correlaciones más bajas con PC1-PC2, aportando a componentes posteriores.
- La oposición `child_mort` vs `gdpp` confirma: países con alto PIB per cápita tienen baja mortalidad infantil.

### Plano principal (PC1 vs PC2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
pca.plano_principal(ax=ax)

# Etiquetar países extremos
coords = pca.row_coords
extremos_pc1 = coords.nlargest(5, 'PC1').index.tolist() + coords.nsmallest(5, 'PC1').index.tolist()
for idx in extremos_pc1:
    ax.annotate(idx, (coords.loc[idx, 'PC1'], coords.loc[idx, 'PC2']),
                fontsize=7, alpha=0.8)

ax.set_title('Plano principal — Países (extremos etiquetados)')
plt.tight_layout()
plt.show()

**Interpretación:**

- Los países desarrollados (alto ingreso, alta esperanza de vida) se ubican en un extremo del PC1; los subdesarrollados en el opuesto.
- En el extremo negativo del PC1 se encuentran países con alta mortalidad infantil (Chad, Sierra Leona, etc.) — candidatos prioritarios para ayuda humanitaria.
- El PC2 separa economías exportadoras/importadoras (ej. países petroleros) del resto.

### Biplot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
pca.biplot(ax=ax)

# Etiquetar algunos extremos
for idx in extremos_pc1[:6]:
    ax.annotate(idx, (coords.loc[idx, 'PC1'], coords.loc[idx, 'PC2']),
                fontsize=7, alpha=0.7, color='darkblue')

ax.set_title('Biplot — Individuos + variables')
plt.tight_layout()
plt.show()

**Interpretación:**

- El biplot confirma que los países con alta mortalidad infantil se alinean con los vectores de `child_mort` y `total_fer`.
- Los países desarrollados apuntan hacia `gdpp`, `income` y `life_expec`.
- La segmentación natural sugiere al menos 3 grupos: desarrollados, en desarrollo y subdesarrollados — base para priorizar intervenciones de Help International.

---
## Tablas de contribuciones y correlaciones

In [ ]:
print('Contribución de cada variable por componente (%):')
print(pca.contribuciones())
print()
print('Cos² de variables (calidad de representación):')
print(pca.cos2_variables())

---
## Interpretación técnica

**Contraste socioeconómico:** Las variables `child_mort` y `gdpp` se oponen diametralmente en el círculo de correlación (correlaciones de signo opuesto en PC1: child_mort ≈ -0.85, gdpp ≈ +0.80). Esto confirma que el PC1 captura el gradiente de desarrollo humano.

Los países en el extremo de la mortalidad infantil alta (visibles en el biplot como los más negativos en PC1) son los candidatos prioritarios para ayuda humanitaria: combinan alta fertilidad, baja esperanza de vida y bajo PIB per cápita.